<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/03-rag/04-why-rag-fails.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG 为什么会失败

**目标：** 诊断糟糕的答案。通常瓶颈在检索质量，而不是生成。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 环境准备

每个 notebook 都是自包含的，接下来的两个单元格会把它从零搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的环境准备 helper，并拉入 `groq` 客户端；本篇还会用到 `requests`、`sentence-transformers`、`numpy`。
2. **载入你的 API key。** 免费 key 在 [console.groq.com](https://console.groq.com/) 领（无需信用卡）。在 Colab 里通过左侧边栏的**钥匙图标** → **Add new secret** 添加，名称必须严格写成 `GROQ_API_KEY`，粘贴值，并把 **Notebook access** 打开。本地运行的话，则把 `GROQ_API_KEY` 设成环境变量。

（完整走查和选模型的指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main" requests sentence-transformers numpy

In [ ]:
from aien import setup

# 载入 GROQ_API_KEY（Colab Secrets 或本地环境变量）并返回一个可直接
# 使用的 Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用
# 报 404，就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 核心论点

当 RAG 系统给出一个糟糕答案时，你的反射是怪模型，然后开始重写 prompt。这个反射通常是错的。在大多数坏掉的 RAG 系统里，模型从没看到过答案：检索悄悄失败了，而生成拿错材料做出了很流畅的东西。

> **⭐ 关键收获 ——** 诊断纪律只有一条规则：**答案错了，先查检索。** 在你碰任何别的东西之前，先看 prompt 里实际装的是什么。

本篇搭一个微小的带标注集合，对着它度量检索，并逐一走完 RAG 失效的四种方式，每一种都针对我们的语料现场演示，每一种都配一个程序化检测法和一个最省事的修法。

## 环境准备：语料、chunk、检索

与 notebook 01 相同的十篇 RFC 语料、section-aware chunk 和暴力向量搜索（单元格重复了一遍，因为这里每个 notebook 都自包含）。


In [ ]:
# 下载共享语料：十篇 IETF RFC，缓存到 data/rfc/。
# 本仓库里每个需要这份语料的 notebook 都包含这个单元格 ——
# 自包含优先于 DRY，好让每个 notebook 都能自己从头跑到尾。
import os
import requests

RFC_NUMBERS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
RFC_TITLES = {
    791: 'IP', 793: 'TCP', 1035: 'DNS', 2616: 'HTTP/1.1', 4271: 'BGP',
    5321: 'SMTP', 6455: 'WebSocket', 6749: 'OAuth 2.0', 7540: 'HTTP/2',
    9110: 'HTTP Semantics',
}
DATA_DIR = 'data/rfc'
os.makedirs(DATA_DIR, exist_ok=True)

corpus = {}  # rfc 编号 -> 原始文本
for num in RFC_NUMBERS:
    path = os.path.join(DATA_DIR, f'rfc{num}.txt')
    if not os.path.exists(path):  # 已缓存：重跑时跳过下载
        resp = requests.get(f'https://www.rfc-editor.org/rfc/rfc{num}.txt', timeout=30)
        resp.raise_for_status()
        with open(path, 'w') as f:
            f.write(resp.text)
    with open(path) as f:
        corpus[num] = f.read()

for num in RFC_NUMBERS:
    print(f'RFC {num:>4}  {RFC_TITLES[num]:<15} {len(corpus[num]):>9,} chars')

In [ ]:
# 清洗 + 按 section 切分的精简复制。notebook 03（分块）
# 才是把它逐步搭起来并解释其权衡的地方；这里只是一个既定条件。
import re

def clean_rfc(text):
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.match(r'^.*\[Page \d+\]\s*$', l)        # 页脚
             and not re.match(r'^RFC \d+\s+.*\S+ \d{4}\s*$', l)]  # 页眉
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()

HEADING_RE = re.compile(r'^\d+(?:\.\d+)*\.?\s+\S', re.MULTILINE)

def chunk_paragraphs(text, max_chars=2400):
    paras = [p for p in text.split('\n\n') if p.strip()]
    chunks, cur = [], ''
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = cur + '\n\n' + p if cur else p
    if cur:
        chunks.append(cur)
    return chunks

def chunk_sections(text, max_chars=2400):
    starts = [m.start() for m in HEADING_RE.finditer(text)]
    if not starts:
        return chunk_paragraphs(text, max_chars)
    chunks = [text[:starts[0]].strip()] if text[:starts[0]].strip() else []
    for a, b in zip(starts, starts[1:] + [len(text)]):
        sec = text[a:b].strip()
        if len(sec) <= max_chars:
            chunks.append(sec)
        else:  # 长 section：按段落打包，并把标题一起带上
            heading = sec.split('\n', 1)[0]
            for sub in chunk_paragraphs(sec, max_chars):
                chunks.append(sub if sub.startswith(heading) else heading + '\n' + sub)
    return [c for c in chunks if c]

chunk_texts, chunk_meta = [], []
for num in RFC_NUMBERS:
    for c in chunk_sections(clean_rfc(corpus[num])):
        chunk_texts.append(c)
        chunk_meta.append({'rfc': num, 'title': RFC_TITLES[num]})
print(f'{len(chunk_texts)} chunks across {len(RFC_NUMBERS)} RFCs')

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunk_texts, normalize_embeddings=True,
                             show_progress_bar=True)

def search(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ q
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), int(i)) for i in top]

def answer(question, k=5, system=None):
    hits = search(question, k)
    context = '\n\n'.join(
        f'[{n + 1}] (RFC {chunk_meta[i]["rfc"]}, {chunk_meta[i]["title"]})\n{chunk_texts[i][:1500]}'
        for n, (_, i) in enumerate(hits)
    )
    prompt = (
        'Answer the question using the numbered passages below. '
        'Cite passages like [1].\n\n'
        f'{context}\n\nQuestion: {question}'
    )
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024, messages=messages,
    )
    return resp.choices[0].message.content

## 一个微小的带标注集合

十个问题，每个都配上包含答案的那篇 RFC，以及一个能标识出「回答该问题的 chunk」的短语。这手写了大约二十分钟，它把「检索感觉还行」变成了一个数字。无论规模大小，标注都胜过感觉，哪怕 n=10。


In [ ]:
labeled = [
    {'q': 'What HTTP status code means the request body is larger than the server will process?',
     'rfc': 9110, 'phrase': 'Content Too Large'},
    {'q': 'Which HTTP status code says the service is temporarily overloaded and the client may retry later?',
     'rfc': 9110, 'phrase': '503'},
    {'q': 'What port does SMTP use for mail transfer between servers?',
     'rfc': 5321, 'phrase': 'port 25'},
    {'q': 'What handshake establishes a TCP connection?',
     'rfc': 793, 'phrase': 'three-way handshake'},
    {'q': 'What GUID does a WebSocket server concatenate with the client key to compute Sec-WebSocket-Accept?',
     'rfc': 6455, 'phrase': '258EAFA5-E914-47DA-95CA-C5AB0DC85B11'},
    {'q': 'In OAuth 2.0, which grant type exchanges an authorization code for an access token?',
     'rfc': 6749, 'phrase': 'authorization_code'},
    {'q': 'Which BGP message type advertises routes to a peer?',
     'rfc': 4271, 'phrase': 'UPDATE message'},
    {'q': 'What does the Time to Live field in the IP header do?',
     'rfc': 791, 'phrase': 'time to live'},
    {'q': 'How does HTTP/2 carry many requests concurrently over one connection?',
     'rfc': 7540, 'phrase': 'multiplexing'},
    {'q': 'What DNS record type holds a host address?',
     'rfc': 1035, 'phrase': 'host address'},
]

def rank_of_correct(item, max_k=50):
    """在 top max_k 之内，来自正确 RFC、且包含答案短语的第一个 chunk 的名次
    （从 1 开始）—— 如果它从未出现，则为 None。
    """
    for rank, (_, i) in enumerate(search(item['q'], max_k), 1):
        if (chunk_meta[i]['rfc'] == item['rfc']
                and item['phrase'].lower() in chunk_texts[i].lower()):
            return rank
    return None

def hit_rate(k):
    ranks = [rank_of_correct(it) for it in labeled]
    return sum(1 for r in ranks if r is not None and r <= k) / len(labeled)

for it in labeled:
    r = rank_of_correct(it)
    print(f'rank {str(r):>4}  RFC {it["rfc"]:>4}  {it["q"][:64]}')
print()
print(f'hit rate @5:  {hit_rate(5):.0%}')
print(f'hit rate @20: {hit_rate(20):.0%}')

跑一下，先读名次这一列，再看汇总数字。每个问题都会落进几个桶之一：名次 1–5（检索没问题）、名次 6–50（检索到了但被埋没）、或 `None`（从未检索到）。这些桶直接对应下面的失效类别，而且注意——你学到这一切时*一个答案都还没生成*。检索指标很便宜；答案评测要花 API 调用。永远先度量便宜的那一层。

## 失效类别 1：答案从未被检索到

**症状：** 名次是 `None`（或比你愿意放进 prompt 的任何名次都更靠后），可答案明明*在*语料里。

**程序化检测：** `rank_of_correct(item) is None`，同时在语料里 grep 那个短语能成功。这一对事实就是指纹。如果 grep 也失败，那你属于类别 3。


In [ ]:
def in_corpus(phrase):
    return any(phrase.lower() in t.lower() for t in chunk_texts)

for it in labeled:
    r = rank_of_correct(it)
    if r is None or r > 5:
        print(f'rank {str(r):>4}  in_corpus={in_corpus(it["phrase"])}  {it["q"][:60]}')

# 两个常见的成因，按需演示：
# (a) 词汇不匹配 —— 查询和回答它的文本没有任何共享词
print()
print('vocabulary-mismatch probe:')
for _, i in search('how do I stop mail from bouncing between servers forever?', 3):
    print(f'   RFC {chunk_meta[i]["rfc"]:>4}  {chunk_texts[i].strip().split(chr(10))[0][:64]}')
print('   (answer: RFC 5321 sec 6.3 Loop Detection -- counting Received: header')
print('    fields. The query says "bouncing forever"; the RFC never does.)')

跑一下，注意在你的索引上有哪些带标注的问题落进这个类别（可能一个都没有）；结果会随 embedding 模型版本略有变化。两个常见的成因：

- **分块把答案切开了。** 标识短语和实质内容落到不同 chunk 里，于是没有哪个单独 chunk 能很好地匹配查询。notebook 03 的边界 demo 正是这个失效的诞生现场。
- **词汇不匹配。** 查询说的是「让邮件别再一直退信」，RFC 说的是通过「统计 Received: header 字段数」来做「detecting loops」。embedding 弥合了其中一部分缺口，但不是全部，而关键词搜索一点都弥合不了。

**最省事的修法：** 第一个成因靠重新分块（更大的 section，或者像我们的 section 分块器那样让每个 chunk 都带上标题，见 notebook 03）；第二个靠混合检索（notebook 02）。两者都是索引侧的修法，完全不涉及 prompt 工程。

## 失效类别 2：检索到了，但被淹没

**症状：** 正确的 chunk 出现在前 20 里，但不在前 5 里，比如名次 8。你只把 5 个 chunk 传给模型，于是它靠那四个半错误的东西作答。即便你传得更多，模型最可靠地关注的也是上下文靠前的部分；一个埋在七个「擦肩而过」下面的答案很容易被忽略。

**程序化检测：** hit@20 为真、hit@5 为假。这两个数字之间的落差*就是*这个失效类别，被度量出来了。


In [ ]:
drowned = [it for it in labeled
           if (r := rank_of_correct(it)) is not None and 5 < r <= 20]

print(f'hit@5 = {hit_rate(5):.0%}, hit@20 = {hit_rate(20):.0%}')
print(f'-> {len(drowned)} question(s) are retrieved-but-drowned:')
for it in drowned:
    print(f'   rank {rank_of_correct(it)}: {it["q"][:64]}')
    print(answer(it['q'], k=5)[:300])   # 看它在没有正确 chunk 的情况下作答
    print()

跑一下，读一读那些被淹没的问题生成的答案：模型拿它得到的那五个 chunk 尽了力，常常产出听起来挺像回事、却引用错了小节的东西——这远比一个明显的失败危险得多。

**最省事的修法：** 重排序（notebook 02）。候选池里已经有答案了；你只需要在切到 5 之前，把前 20 排得更好。把 k 从 5 加宽到 20 也「管用」，但为了修一个排序问题，每个查询都要多花 4 倍上下文 token，而在规模上 reranker 更便宜。

## 失效类别 3：根本不在语料里，最安静的那种

**症状：** 没有症状，而这正是问题所在。问一件语料没覆盖的事，检索会忠实地返回最近的 chunk（notebook 01 的陷阱 3：最近邻总会返回*某个东西*），而模型——它从预训练里就知道答案——会写出一段自信、听起来正确的回复，尽管你的语料从未支持过它。用户学会信任它。然后它会在一个预训练内容有误或过时的提问上做同样的事，语气同样自信。

PKCE 是 RFC 7636（2015），一个在 RFC 6749 *之后*发布的 OAuth 2.0 扩展，不在这十篇里（`PKCE` 这个字符串在语料中任何地方都不出现；用 `any('pkce' in t.lower() for t in chunk_texts)` 核实）。模型当然从预训练里熟得不能再熟。看：


In [ ]:
q = 'How does PKCE protect the OAuth authorization code flow?'

print('top-3 retrieval scores (nothing relevant exists):')
for score, i in search(q, 3):
    print(f'  {score:.3f}  RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})')
print()
print('--- naive RAG answer:')
print(answer(q)[:600])

跑一下，注意两件事。第一，检索看起来*很健康*：它返回带体面分数的 OAuth chunk，因为 PKCE 问题和 RFC 6749 的材料在语义上很接近，所以分数不会为此报警。第二，答案大概*是对的*（模型从预训练里知道 PKCE），但它引用了一些其实并不支持该说法的段落，或者含含糊糊地朝它们挥挥手。正确但无据可依，仍然是一种失败：你建 RAG 恰恰就是为了让答案落地在*你的*文档上，而这个答案落地在你控制不了的任何东西上。等它碰上一个预训练内容有误或过时的提问时，它会以同样自信的方式失败。

**程序化检测：** 很难，而这正是要点。实用的检测器有：对问题里的关键实体做语料 grep（「`PKCE` 这个字面字符串到底有没有出现过？」）、引用支撑检查（被引用的段落真的包含那个说法吗？），或者一个包含已知无法回答问题的评测集，并期望模型答「我不知道」。

**最省事的修法：** 告诉模型，拒答是一个可接受的答案，并让这条指令具体：


In [ ]:
GROUNDED = ('Answer ONLY from the provided passages. If the passages do not '
            'contain the information needed, reply exactly: "The corpus does '
            'not cover this." Do not use outside knowledge, even if you know '
            'the answer.')

print(answer(q, system=GROUNDED)[:400])
print()
# 再确认这条指令不会弄坏可回答的问题：
print(answer('What port does SMTP use for mail transfer between servers?',
             system=GROUNDED)[:400])

两个都跑，检查一下：PKCE 问题现在应该被拒答，而 SMTP 问题仍应被回答。第二个检查很重要。一条过于激进的拒答指令会让模型对可回答问题也拒答，那是一种回归，而你只有重跑那批可回答的问题才能抓到它。（你已经开始体会到第 04 节为什么存在了。）

## 失效类别 4：检索正确，但问题需要多跳

**症状：** 问题需要来自*两篇文档*的事实，而单次查询的前 k 会聚集在查询措辞更偏袒的那篇文档周围。每个单独的 chunk 都被「正确」检索到了。只是不存在某个单独的 chunk、或单独的邻域，能覆盖这个对比。


In [ ]:
q = ('How do SMTP and HTTP each tell a client that a failure is temporary '
     'and the request should be retried later?')

print('single-query top-5 -- count RFCs represented:')
for score, i in search(q, 5):
    print(f'  {score:.3f}  RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})')

跑一下，数一数前 5 里有几篇 RFC：通常一种协议占主导，另一种几乎不出现，于是用这些命中生成的答案很好覆盖了问题的一半，另一半靠现编。

**程序化检测：** 问题里点了两个实体、而它的前 k 被单一来源主导，例如问题提到两种协议时 `len({meta['rfc'] for hits}) == 1`。很粗糙，但能标出这个类别。

**最省事的修法：** 查询分解。把问题拆成子查询（手工拆，或一次便宜的模型调用），分别检索，生成之前再合并：


In [ ]:
sub_queries = [
    'How does SMTP indicate a temporary failure that the client should retry?',
    'How does HTTP indicate a temporary failure that the client should retry?',
]

merged, seen = [], set()
for sq in sub_queries:
    for score, i in search(sq, 3):
        if i not in seen:
            seen.add(i)
            merged.append((score, i))

print('merged hits -- both protocols now represented:')
for score, i in merged:
    print(f'  {score:.3f}  RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})')

context = '\n\n'.join(
    f'[{n + 1}] (RFC {chunk_meta[i]["rfc"]}, {chunk_meta[i]["title"]})\n{chunk_texts[i][:1500]}'
    for n, (_, i) in enumerate(merged)
)
resp = client.chat.completions.create(
    model=MODEL, max_tokens=1024,
    messages=[{'role': 'user', 'content':
               f'Answer using only the passages, citing like [1].\n\n{context}\n\nQuestion: {q}'}],
)
print()
print(resp.choices[0].message.content[:700])

跑一下，把合并后的命中列表和单次查询的对比：两篇 RFC 都在了，生成的对比现在能引用双方。分解是最便宜的多跳修法；agentic 检索（让模型自己发起后续查询，属于第 05 节的地盘）是同一想法昂贵的通用版。

## 诊断卡片

当答案是错的，按顺序过这张清单。它的排序依据是「检查」与「修复」两件事各自的便宜程度：

| # | 类别 | 检测 | 最省事的修法 |
|---|---|---|---|
| 1 | 答案未被检索到 | 名次为 `None`，但语料 grep 成功 | 重新分块 / 混合检索 |
| 2 | 检索到了但被淹没 | hit@20 ✓、hit@5 ✗ | 对候选池重排序 |
| 3 | 不在语料里 | 语料 grep 失败；引用不支持该说法 | 说「我不知道」的指令 + 无法回答问题的评测 |
| 4 | 多跳 | 两实体问题上前 k 被单一来源主导 | 查询分解 |

最后做一个诚实的自查：在 10 个手写问题上的命中率只是一个**原型评测**。它把感觉变成了数字，并告诉了我们流水线在哪里断裂，但 n=10 分不清一个 78% 的系统和 85% 的系统，它不给答案*质量*打分，而且那些生成结果是我们用眼睛看的、不是评出来的。第 04 节拿走的正是这个产物（问题、标注、命中率），并把它做严谨：更大的黄金集、用于答案质量的 LLM-as-judge，以及回归评测——让这个数字变成一道关卡，而不是一张快照。

## 框架在这里的位置：LangChain 与 LlamaIndex

你用最原始的零件搭出了检索、混合 + 重排序、分块和失效诊断：numpy 里的 embedding、一个 `search()` 函数、你能读懂的那些 prompt。这是刻意安排的。**RAG 是一种模式，不是一个库**，而现在你能看清任何 RAG 框架在其抽象层下到底在做什么。

**LangChain** 和 **LlamaIndex** 是你在招聘帖上最常看到的两个。它们的内核接的正是你搭过的*同一条* retrieve → augment → generate 循环，另外加上：

- **连接器：** 面向 PDF、Notion、Slack、SQL、网页的 loader；几十种向量存储（Pinecone、Chroma、pgvector）藏在一个接口后面。这才是真正省时间的地方，因为你不用再写摄入的胶水代码了。
- **可组合的流水线：** retriever → reranker → prompt → LLM 作为可互换的组件（LangChain 的 LCEL、LlamaIndex 的 query engine），而不是手接线的一串函数。
- **预制策略：** 你刚刚手写的多查询与子问题检索，出厂就是 `MultiQueryRetriever` / 子问题 query engine；parent-document 和 sentence-window 分块也现成。
- **Agentic RAG：** 把查询路由到合适的索引，或者让一个 agent（第 05 节）决定*何时*检索，都替你接好了。

> **⭐ 关键收获 ——** 框架改变的是*你怎样组装* RAG，而不是*RAG 是什么*。本节里的每一种失效模式（什么都没检索到、检索到了但被淹没、不在语料里、多跳）在 LangChain 和 LlamaIndex **内部**同样会发生，而它们不会替你去诊断。你要用在这里建立的同一副「检索质量」眼镜去 debug 它们。

**什么时候用它：** 当连接器/向量存储的广度能省下真实的胶水代码时，或者你的团队已经标准化在它上面时。**什么时候不用：** 一条专注的生产流水线，写成你写的这约 40 行，往往比堆一层抽象*更清晰*——何况检索质量一下降，你还得去逆向那层抽象。无论哪条路，你都能把框架评估得很好，因为你知道它在抽象什么。

> **🔵 面试信号 ——** 「我用过 LangChain，但我还能解释它底层做了什么、以及什么时候裸检索是更好的选择」，这比「我们用了 LangChain」*或*「我永远不用框架」都要强得多。流利，加上判断力。

## 练习

1. **扩充带标注集合。** 自己再加 5 个问题，其中至少有一个是你*预期*会失败的（一个精确标识符、一个没有关键词重叠的同义改写），以及一个从语料无法回答的。重新计算 hit@5 和 hit@20。那些失败落进了你预测的类别里吗？
2. **在带标注集合上对比混合检索与向量检索。** 把 `bm25_search` 和 `rrf` 从 notebook 02 移植到本篇，对所有带标注问题计算「仅向量」与「混合」的 hit@5。你现在有了第一批真实证据，能判断混合检索在这份语料上是否值回成本：一个数字，而不是一次猜测。
3. **引用支撑检查器。** 用一次模型调用写一个 `supports(claim_answer, cited_chunks)`：被引用的文本真的包含所给答案吗？在类别 3 的 PKCE 答案上跑一遍，再在三个可回答问题跑一遍。这是 groundedness 评测的种子。
4. **自动失效分诊。** 用 `rank_of_correct` 和 `in_corpus` 写一个 `triage(item)`，返回 `'ok'`、`'not_retrieved'`、`'drowned'` 或 `'not_in_corpus'`。在完整的带标注集合上跑一遍，打印类别直方图。你把诊断卡片写成了函数——而这正是第 04 节期望你给出的形态。
